<a href="https://colab.research.google.com/github/ga426553-sudo/Classification-of-Breast-Cancer-Subtypes-machine-learning---CuMiDa-22820/blob/main/KNN_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Tercer código - Cáncer de Mama 🌸

**KNN y Evaluation**

In [ ]:
# ============================================
# CÓDIGO 3: KNN Y EVALUACIÓN
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("🔬 CÓDIGO 3: EVALUACIÓN DE KNN")
print("="*60)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔬 CÓDIGO 3: EVALUACIÓN DE KNN


###Cargar datos 🌺

In [ ]:
# 1. CARGAR DATOS
# ================
print(f"\n📂 Cargando datos del Código 2...")

with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/subsets.pkl', 'rb') as f:
    subsets = pickle.load(f)

y_train = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_final.npy')
y_test = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_test_final.npy')

print(f"✅ Datos cargados:")
print(f"   Subconjuntos: {list(subsets.keys())}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test: {y_test.shape}")


📂 Cargando datos del Código 2...
✅ Datos cargados:
   Subconjuntos: ['EO', 'BBA', 'CSA', 'RDA', 'GA']
   y_train: (206,)
   y_test: (28,)


###Configuración del KNN 🌺

In [ ]:
# 2. CONFIGURAR KNN
# ==================
knn_params = {
    'n_neighbors': 5,
    'weights': 'uniform',
    'algorithm': 'auto',
    'leaf_size': 30,
    'p': 2,
    'metric': 'minkowski'
}

knn_model = KNeighborsClassifier(**knn_params)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

###Evaluar cada subconjunto 🌺

In [ ]:
# 3. EVALUAR CADA SUBCONJUNTO
# ============================
results = {}

for name in subsets.keys():
    print(f"\n{'─'*40}")
    print(f"🔍 Subconjunto: {name} - Genes: {subsets[name]}")

    X_train_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/train_{name}.npy')
    X_test_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/test_{name}.npy')

    # Validación cruzada
    cv_acc = cross_val_score(knn_model, X_train_subset, y_train, cv=cv, scoring='accuracy')
    cv_f1 = cross_val_score(knn_model, X_train_subset, y_train, cv=cv, scoring='f1')
    cv_auc = cross_val_score(knn_model, X_train_subset, y_train, cv=cv, scoring='roc_auc')

    # Evaluación en test
    knn_model.fit(X_train_subset, y_train)
    y_pred = knn_model.predict(X_test_subset)
    y_proba = knn_model.predict_proba(X_test_subset)[:, 1]

    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_proba)

    results[name] = {
        'cv_accuracy': f"{cv_acc.mean():.4f} ± {cv_acc.std():.4f}",
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'test_auc': test_auc,
        'genes': subsets[name]
    }

    print(f"   CV Accuracy: {results[name]['cv_accuracy']}")
    print(f"   Test Accuracy: {test_acc:.4f}")
    print(f"   Test F1: {test_f1:.4f}")
    print(f"   Test AUC: {test_auc:.4f}")


────────────────────────────────────────
🔍 Subconjunto: EO - Genes: ['NM_002644', 'BM511013', 'NM_182611']
   CV Accuracy: 1.0000 ± 0.0000
   Test Accuracy: 1.0000
   Test F1: 1.0000
   Test AUC: 1.0000

────────────────────────────────────────
🔍 Subconjunto: BBA - Genes: ['NM_002644', 'BM511013']
   CV Accuracy: 0.9950 ± 0.0150
   Test Accuracy: 0.9286
   Test F1: 0.9615
   Test AUC: 0.9808

────────────────────────────────────────
🔍 Subconjunto: CSA - Genes: ['BM511013', 'NM_182611']
   CV Accuracy: 0.9950 ± 0.0150
   Test Accuracy: 0.9643
   Test F1: 0.9804
   Test AUC: 1.0000

────────────────────────────────────────
🔍 Subconjunto: RDA - Genes: ['NM_002644']
   CV Accuracy: 0.9902 ± 0.0195
   Test Accuracy: 0.8929
   Test F1: 0.9434
   Test AUC: 0.6827

────────────────────────────────────────
🔍 Subconjunto: GA - Genes: ['AW015426']
   CV Accuracy: 0.9950 ± 0.0150
   Test Accuracy: 1.0000
   Test F1: 1.0000
   Test AUC: 1.0000


###Mostrar resumen de los resultados 🌺

In [ ]:

# 4. MOSTRAR RESULTADOS
# ======================
print(f"\n{'='*60}")
print("📊 RESULTADOS FINALES")
print('='*60)

for name, r in results.items():
    print(f"\n{name}:")
    print(f"   Genes: {r['genes']}")
    print(f"   CV Accuracy: {r['cv_accuracy']}")
    print(f"   Test Accuracy: {r['test_accuracy']:.4f}")
    print(f"   Test F1: {r['test_f1']:.4f}")
    print(f"   Test AUC: {r['test_auc']:.4f}")


📊 RESULTADOS FINALES

EO:
   Genes: ['NM_002644', 'BM511013', 'NM_182611']
   CV Accuracy: 1.0000 ± 0.0000
   Test Accuracy: 1.0000
   Test F1: 1.0000
   Test AUC: 1.0000

BBA:
   Genes: ['NM_002644', 'BM511013']
   CV Accuracy: 0.9950 ± 0.0150
   Test Accuracy: 0.9286
   Test F1: 0.9615
   Test AUC: 0.9808

CSA:
   Genes: ['BM511013', 'NM_182611']
   CV Accuracy: 0.9950 ± 0.0150
   Test Accuracy: 0.9643
   Test F1: 0.9804
   Test AUC: 1.0000

RDA:
   Genes: ['NM_002644']
   CV Accuracy: 0.9902 ± 0.0195
   Test Accuracy: 0.8929
   Test F1: 0.9434
   Test AUC: 0.6827

GA:
   Genes: ['AW015426']
   CV Accuracy: 0.9950 ± 0.0150
   Test Accuracy: 1.0000
   Test F1: 1.0000
   Test AUC: 1.0000
